In [6]:
# =============================================================================
# STAGE 6 — Model Building & Training
# -----------------------------------------------------------------------------
# Project: Student Performance Prediction & Early Intervention (#23)
# Team   : Abhi | Aparna | Jia  |  Predictive Analytics 2025-26
# Run    : python stage6_model_building.py   (after stage5)
# Input  : ../data/student_features.csv
#          ../data/selected_features.txt
# Output : ../models/  (trained model files for Abhi's stages)
# =============================================================================
import sys
!{sys.executable} -m pip install imbalanced-learn

import os
import warnings
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)


# ── Load from Stage 5 output ──────────────────────────────────────────────────
def load_data():
    feat_path = "../data/student_features.csv"
    sel_path  = "../data/selected_features.txt"

    if not os.path.exists(feat_path):
        raise FileNotFoundError(
            "student_features.csv not found.\n"
            "Please run stage5_feature_engineering.py first."
        )

    df = pd.read_csv(feat_path)
    with open(sel_path) as f:
        selected_features = [line.strip() for line in f if line.strip()]

    X = df[selected_features]
    y = df["pass"]
    print(f"✅  Loaded student_features.csv  →  X: {X.shape}  y: {y.shape}")
    print(f"    Features: {selected_features}\n")
    return X, y, selected_features


# ── 6.1  Train-Test Split ─────────────────────────────────────────────────────
def split_data(X, y):
    print("=" * 55)
    print("  6.1  TRAIN / TEST SPLIT  (80 / 20, stratified)")
    print("=" * 55)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"  Train : {X_train.shape[0]} samples  |  "
          f"class dist: {y_train.value_counts().to_dict()}")
    print(f"  Test  : {X_test.shape[0]} samples   |  "
          f"class dist: {y_test.value_counts().to_dict()}")
    return X_train, X_test, y_train, y_test


# ── 6.2  SMOTE — Handle Class Imbalance ──────────────────────────────────────
def apply_smote(X_train, y_train):
    print("\n" + "=" * 55)
    print("  6.2  SMOTE — HANDLE CLASS IMBALANCE")
    print("=" * 55)
    print(f"  Before SMOTE: {pd.Series(y_train).value_counts().to_dict()}")
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    print(f"  After  SMOTE: {pd.Series(y_res).value_counts().to_dict()}")
    print(f"  Training set size: {X_train.shape[0]} → {X_res.shape[0]}")
    return X_res, y_res


# ── 6.3  Feature Scaling ──────────────────────────────────────────────────────
def scale_features(X_train_sm, X_test):
    print("\n" + "=" * 55)
    print("  6.3  FEATURE SCALING  (StandardScaler)")
    print("=" * 55)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sm)   # fit on train only
    X_test_scaled  = scaler.transform(X_test)            # transform test
    print("  ✅  Scaler fit on training data, applied to test data.")
    return X_train_scaled, X_test_scaled, scaler


# ── 6.4  Define & Train Models ────────────────────────────────────────────────
def train_models(X_train_scaled, y_train_sm):
    print("\n" + "=" * 55)
    print("  6.4  MODEL TRAINING")
    print("=" * 55)

    model_defs = {
        "Decision Tree": DecisionTreeClassifier(
            max_depth=5, min_samples_split=10,
            min_samples_leaf=5, random_state=42
        ),
        "Logistic Regression": LogisticRegression(
            C=1.0, max_iter=1000, random_state=42
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1,
            max_depth=4, random_state=42
        ),
    }

    trained = {}
    for name, model in model_defs.items():
        model.fit(X_train_scaled, y_train_sm)
        trained[name] = model
        print(f"  ✅  {name} trained.")

    return trained


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    X, y, selected_features = load_data()
    X_train, X_test, y_train, y_test = split_data(X, y)
    X_train_sm, y_train_sm = apply_smote(X_train, y_train)
    X_train_scaled, X_test_scaled, scaler = scale_features(X_train_sm, X_test)
    trained_models = train_models(X_train_scaled, y_train_sm)

    # ── Save everything for Abhi (Stages 7, 8, 9) ────────────────────────────
    best_model = trained_models["Gradient Boosting"]
    joblib.dump(trained_models,    f"{MODELS_DIR}/trained_models.pkl")
    joblib.dump(best_model,        f"{MODELS_DIR}/gradient_boosting_model.pkl")
    joblib.dump(scaler,            f"{MODELS_DIR}/scaler.pkl")
    joblib.dump(selected_features, f"{MODELS_DIR}/selected_features.pkl")

    # Save test split so Abhi can evaluate on the same hold-out set
    test_df = pd.DataFrame(X_test_scaled, columns=selected_features)
    test_df["pass"] = y_test.values
    test_df.to_csv("../data/test_set.csv", index=False)

    # Save train_scaled so Abhi can pass it to LIME
    train_df = pd.DataFrame(X_train_scaled, columns=selected_features)
    train_df["pass"] = y_train_sm
    train_df.to_csv("../data/train_scaled.csv", index=False)

    print(f"\n✅  Stage 6 complete.")
    print(f"    Saved models    : {MODELS_DIR}/")
    print(f"    Saved test set  : ../data/test_set.csv")
    print(f"    Saved train set : ../data/train_scaled.csv")
    print("    Next  : python stage7_evaluation.py  (Abhi)")



   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅  Loaded student_features.csv  →  X: (395, 10)  y: (395,)
    Features: ['avg_grade', 'G2', 'Fedu', 'paid', 'absences', 'G1', 'Fjob', 'famsize', 'Dalc', 'reason']

  6.1  TRAIN / TEST SPLIT  (80 / 20, stratified)
  Train : 316 samples  |  class dist: {0: 222, 1: 94}
  Test  : 79 samples   |  class dist: {0: 56, 1: 23}

  6.2  SMOTE — HANDLE CLASS IMBALANCE
  Before SMOTE: {0: 222, 1: 94}
  After  SMOTE: {0: 222, 1: 222}
  Training set size: 316 → 444

  6.3  FEATURE SCALING  (StandardScaler)
  ✅  Scaler fit on training data, applied to test data.

  6.4  MODEL TRAINING
  ✅  Decision Tree trained.
  ✅  Logistic Regression trained.
  ✅  Gradient Boosting trained.

✅  Stage 6 complete.
    Saved models    : ../models/
    Saved test set  : ../data/test_set.csv
    Saved train set : ../data/train_scaled.csv
    Next  : python stage7_evaluation.py  (Abhi)
